# CityFlow: Task 1 and Task 2

**Source:** https://www.kaggle.com/datasets/mobeenfatimah/cityflow-smart-urban-mobility-and-traffic-iot

**Selected target:** `emission_estimate`

**Important limitation:** The dataset contains hourly observations and a timestamp. It is a time-series dataset, which conflicts with the exercise instruction not to select time-series data. This notebook analyses the requested dataset, but instructor approval or a replacement regression dataset is needed for final submission.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_CANDIDATES = [
    Path('data/cityflow/smart_city_traffic_mobility.csv'),
    Path('lecture_tasks/data/cityflow/smart_city_traffic_mobility.csv'),
    Path('../data/cityflow/smart_city_traffic_mobility.csv'),
]
DATA_PATH = next(path for path in DATA_CANDIDATES if path.exists())
REPORT_DIR = DATA_PATH.parents[2] / 'reports'
REPORT_DIR.mkdir(exist_ok=True)
TARGET = 'emission_estimate'

df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df.head()

## Step 1a: Datatypes

In [ ]:
dtype_report = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'unique_values': df.nunique(dropna=False),
})
dtype_report

The timestamp was converted to datetime because date columns should not remain strings during analysis. No other values are changed at this stage.

## Step 1b: Shape and time-series check

In [ ]:
print('Shape:', df.shape)
print('Time range:', df['timestamp'].min(), 'to', df['timestamp'].max())
print('Unique timestamps:', df['timestamp'].nunique())
print('Rows per intersection, selected values:')
display(df.groupby('intersection_id').size().describe())

The large row count is useful for statistical analysis, but the timestamp and repeated sensor observations mean that ordinary random train/test splitting could leak temporal or intersection-specific information. This is also the reason the dataset may not satisfy the exercise restriction.

## Step 1c: Missing values, duplicates, zeroes, and suspicious values

In [ ]:
print('Total missing values:', int(df.isna().sum().sum()))
display(df.isna().sum().sort_values(ascending=False).head(15))
print('Complete duplicate rows:', int(df.duplicated().sum()))

numeric_columns = df.select_dtypes(include='number').columns
zero_report = (df[numeric_columns] == 0).sum().sort_values(ascending=False)
display(zero_report.to_frame('zero_count'))

display(df[numeric_columns].describe().T)

Zeroes must be interpreted using the data dictionary. For example, binary event indicators can legitimately contain many zeroes. A zero in an amount or measurement column may require more investigation, but it is not automatically missing.

## Step 1d: Target and categorical distributions

In [ ]:
print('Target summary:', TARGET)
display(df[TARGET].describe())

plt.figure(figsize=(9, 5))
sns.histplot(data=df, x=TARGET, bins=40, kde=True)
plt.title('Distribution of emission_estimate')
plt.tight_layout()
plt.show()

categorical_columns = df.select_dtypes(include=['object', 'category']).columns
for column in categorical_columns:
    print(f'\n{column}: {df[column].nunique()} unique values')
    display(df[column].value_counts(dropna=False).head(15))

## Preliminary issue list

- The data is time-series data, so it conflicts with the assignment restriction.
- The timestamp and repeated intersection observations may require a time-aware validation strategy later.
- The target distribution should be checked for skewness and extreme emissions.
- High-cardinality identifier columns such as record and road identifiers may need special handling.
- Many zeroes may be valid binary event indicators rather than missing values.
- Profiling alerts must be verified against the meaning of each sensor variable before any cleaning decision.

## Generate ydata-profiling report

In [ ]:
try:
    from ydata_profiling import ProfileReport
except ImportError:
    from data_profiling import ProfileReport

profile = ProfileReport(
    df,
    title='CityFlow Smart Urban Mobility Preliminary EDA',
    explorative=True
)
profile.to_file(REPORT_DIR / 'cityflow_ydata_profile.html')